In [1]:
# ==========================================
# 1. Environment Setup & Load Cleaned Data
# ==========================================

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import joblib

# 1. โหลดข้อมูลที่ Cleaning แล้ว
df = pd.read_csv('enriched_crop_dataset_cleaned.csv')

# 2. แยก Features (X) และ Target (y)
X = df.drop(columns=['label'])
y = df['label']

# แปลงชื่อพืชเป็นตัวเลขสำหรับรองรับ XGBoost
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print("จำนวน Features หลังทำ Encoding:", X.shape[1])

จำนวน Features หลังทำ Encoding: 11


In [ ]:
# ==========================================
# 2. Data Splitting & Feature Scaling & Feature Engineering/Data Transformation (one-hot encoding) 
# ==========================================

In [4]:
# แปลงคอลัมน์ Categorical/Text ใน X ให้เป็นตัวเลข (One-Hot Encoding)
X = pd.get_dummies(X, drop_first=True)

# แบ่งข้อมูล 80/20 พร้อมคงสัดส่วน Class เป้าหมาย
# 3. แบ่งข้อมูล Train / Test (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)
#ในการแบ่งขข้อมูล ความหมายของแต่ละอันคือ จะแบ่งเป็น 3 phase Training Phase, Testing Phase, Evaluation Phase
#1. x_train, y_train 2 อันนี้คือข้อมูลตอน Train เปรียบเสมือนให้ทำโจทย์และมีเฉลย -> model.fit(X_train_scaled, y_train)
#2. x_test เปรียบเหมือนตอนทำข้อสอบแล้วผลลัพธ์จะถูกเก็บไว้ในตัวแปรใหม่ y_pred -> y_pred = model.predict(X_test_scaled)
#3. y_test คือคำตอบที่ถูกต้องที่เราซ่อนไว้ของข้อสอบที่เราทำในข้อ 2 ว่าตอบถูกกี่ข้อ ตอบผิดกี่ข้อ เพื่อคำนวณเป็นค่าความแม่นยำ -> acc = accuracy_score(y_test, y_pred)
# 4. Feature Scaling (ช่วยให้โมเดลระยะทางอย่าง KNN ทำงานได้ดีขึ้น)
#คือเราจะไม่ปรับสเกลก็ได้แต่มันจะไม่เป็นธรรมต่อบาง Model ในการวัดประสิทธิที่ เช่น Model ที่ใช้สูตรในการคำนวณระยะทางอย่าง KNN ผมเลยตัดสินใจในการ Train แต่ละModel ให้ใช้ X_Train_scale แทน X_Train
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"จำนวนข้อมูล Train: {X_train.shape[0]} แถว | Test: {X_test.shape[0]} แถว")

จำนวนข้อมูล Train: 1760 แถว | Test: 440 แถว


In [5]:
# ==========================================
# 3. Model Training
# ==========================================

In [6]:
# 5. นำเข้าโมเดลยอดนิยมสำหรับ Classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

models = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(random_state=42, eval_metric='mlogloss'),
    "Gaussian Naive Bayes": GaussianNB(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "K-Nearest Neighbors": KNeighborsClassifier()
}

print("เตรียมโมเดล Random Forest, XGBoost, Gaussian Naive Bayes, Decision Tree, และ K-Nearest Neighbors เสร็จสิ้น")

เตรียมโมเดล Random Forest, XGBoost, Gaussian Naive Bayes, Decision Tree, และ K-Nearest Neighbors เสร็จสิ้น


In [7]:
# ==========================================
# 4. Model Evaluation & Comparison
# ==========================================

In [13]:
# 6. ทดสอบและเปรียบเทียบประสิทธิภาพของแต่ละโมเดล
results = []

for name, model in models.items():
    # ประเมินผลด้วย 5-Fold Cross Validation
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy')
    
    # ฝึกโมเดลและทดสอบบน Test Set โดยใช้ X_train_scaled
    #ไม่ส่งผลเสียต่อ Tree-based Models: โมเดลกลุ่มต้นไม้ (Decision Tree, Random Forest, XGBoost) ตัดสินใจจากเกณฑ์การแบ่งข้อมูล (เช่น if Feature_A <= 1.5) การเปลี่ยนสเกลข้อมูลไม่ได้เปลี่ยนลำดับหรือโครงสร้างการตัดแบ่งกิ่ง ผลลัพธ์จึงออกมาเท่าเดิมหรือแทบไม่ต่างกันเลย
    #ช่วยให้ Distance-based Models ทำงานได้เต็มประสิทธิภาพ: โมเดลอย่าง KNN (หรือ SVM / Logistic Regression หากนำมาเพิ่มในอนาคต) บังคับว่าต้องใช้ข้อมูลที่ Scale แล้วเท่านั้น 
    # ไม่งั้นคอลัมน์ค่าหลักร้อยอย่าง rainfall จะกลบความสำคัญของคอลัมน์อื่นหมด
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    test_acc = accuracy_score(y_test, y_pred)
    
    results.append({
        "Model": name,
        "CV Mean Accuracy (%)": round(cv_scores.mean() * 100, 2),
        "Test Accuracy (%)": round(test_acc * 100, 2)
    })

# แสดงผลเปรียบเทียบเรียงตาม Test Accuracy
results_df = pd.DataFrame(results).sort_values(by="Test Accuracy (%)", ascending=False)
print("--- ตารางเปรียบเทียบความแม่นยำของแต่ละ Model ---")
print(results_df.to_string(index=False))

--- ตารางเปรียบเทียบความแม่นยำของแต่ละ Model ---
               Model  CV Mean Accuracy (%)  Test Accuracy (%)
       Random Forest                 98.64              99.77
             XGBoost                 99.03              99.09
       Decision Tree                 98.52              97.73
Gaussian Naive Bayes                 43.12              45.45
 K-Nearest Neighbors                 30.57              36.36


In [9]:
# ==========================================
# 5. Model Dump / Export Pipeline (เลือก Model ที่ดีที่สุด)
# ==========================================

In [14]:
# 1. สร้างและ Fit โมเดล XGBoost
best_model = XGBClassifier(random_state=42, eval_metric='mlogloss')
best_model.fit(X_train_scaled, y_train)

# 2. ทดสอบทำนายผลบน Test Set
y_pred = best_model.predict(X_test_scaled)
print(f"=== Test Accuracy (XGBoost): {accuracy_score(y_test, y_pred) * 100:.2f}% ===\n")

# 3. รวบรวมวัตถุที่จำเป็นสำหรับการทำนายสภาพแวดล้อมใหม่ไว้ใน dict
pipeline_artifacts = {
    'model': best_model,                        # โมเดล XGBoost ที่เทรนแล้ว
    'scaler': scaler,                          # Scaler สำหรับปรับสเกลข้อมูล
    'label_encoder': le,                       # LabelEncoder สำหรับแปลงตัวเลขกลับเป็นชื่อพืช
    'feature_names': list(X.columns)           # ดึงรายชื่อ Features ทั้งหมดแบบ Dynamic
}

# Dump เป็นไฟล์ .pkl สำหรับนำไปสร้างแอปพลิเคชัน
# enriched_crop_pipeline.pkl ใช้สำหรับการทำนายพืชที่เหมาะสมกับสภาพแวดล้อมใหม่ จาก 22 ชนิดพืชที่มีใน Dataset
joblib.dump(pipeline_artifacts, 'enriched_crop_pipeline.pkl')
print("Dump โมเดลและ Artifacts สำเร็จเป็นไฟล์: 'enriched_crop_pipeline.pkl'")

=== Test Accuracy (XGBoost): 99.09% ===

Dump โมเดลและ Artifacts สำเร็จเป็นไฟล์: 'enriched_crop_pipeline.pkl'


In [11]:
# ==========================================
# 6. Verification Test (ทดสอบโหลดโมเดลกลับมาใช้งาน)
# ==========================================

In [15]:
# 1. โหลด Pipeline และ Artifacts กลับมาทดสอบ
loaded_pipeline = joblib.load('enriched_crop_pipeline.pkl')
test_model = loaded_pipeline['model']
test_scaler = loaded_pipeline['scaler']
test_le = loaded_pipeline['label_encoder']

# 2. ดึงข้อมูลตัวอย่าง 15 แถวแรกจาก X_test_scaled และ y_test
X_sample_raw = X_test.iloc[:15].copy() # ค่าปัจจัยดิบ N, P, K...
X_sample_scaled = X_test_scaled[:15]
y_sample_actual_num = y_test[:15]


# ดึงค่า Location จาก df ต้นฉบับอ้างอิงตาม Index ของข้อมูลสุ่ม
# (หากใน df ใช้ชื่ออื่น เช่น 'state', 'city', 'district' ให้เปลี่ยนชื่อคอลัมน์ตรงนี้ให้ตรงกัน)
sample_locations = df.loc[X_sample_raw.index, 'location'].values

# 3. ทำนายผลชนิดพืช และคำนวณความมั่นใจ (Confidence Score)
#ป้อนโจทย์ 15 แถวให้โมเดลประมวลผล
y_sample_pred_num = test_model.predict(X_sample_scaled)
#คำนวณความมั่นใจของโมเดล (Confidence Score) โดยใช้ predict_proba เพื่อดึงค่า Probability ในแต่ละแถวของพืช 22 ชนิด
sample_proba = test_model.predict_proba(X_sample_scaled)
#จากตารางความน่าจะเป็น 22 คอลัมน์ ให้ดึงเฉพาะ "ค่าความน่าจะเป็นที่สูงที่สุด" ของแต่ละแถวออกมา ซึ่งค่านี้จะถูกใช้เป็น คะแนนความมั่นใจ (Confidence Score) ว่าโมเดลมั่นใจในคำตอบที่เลือกกี่ %
confidence_scores = np.max(sample_proba, axis=1)

# 4. แปลงรหัสตัวเลขกลับเป็นชื่อพืช (Text)
#นำรหัสตัวเลขเฉลยจริง (y_sample_actual_num) แปลงย้อนกลับเป็น ข้อความชื่อพืชจริง เช่น เปลี่ยน 0 -> 'apple'
actual_crops = test_le.inverse_transform(y_sample_actual_num)
#เหมือน Actual_Crop แต่นี่แปลงเลขจากการทำนายของโมเดล ว่าโมเดลทำนายว่าเป็นพืชชนิดใด เช่น เปลี่ยน 0 -> 'apple' และนำไปเปรียบเทียบกับ Actual_Crop ว่าตรงกันหรือไม่ จะได้ผลลัพธ์เป็น True/False ในช่อง Match
predicted_crops = test_le.inverse_transform(y_sample_pred_num)

# 5. แสดงผลลัพธ์เปรียบเทียบ
results_df = pd.DataFrame({
    'Sample_No': range(1, 16),
    'Excel_No': X_sample_raw.index + 2,                 # +2 เพราะ Excel เริ่มนับแถวจาก 1 และมี Header
    'Location': sample_locations,                       # เพิ่มคอลัมน์แสดงสถานที่
    'Actual_Crop': actual_crops,                        # ชื่อพืชจริง
    'Predicted_Crop': predicted_crops,                  # ชื่อพืชที่โมเดลทำนาย
    'Confidence (%)': (confidence_scores * 100).round(2), # ความมั่นใจของโมเดล
    'Match': actual_crops == predicted_crops            # ทำนายถูกต้องหรือไม่ (True/False)
})

print("\n--- ผลการทดสอบทำนายข้อมูลตัวอย่าง 15 แถวแรก ---")
print(results_df.to_string(index=False))


--- ผลการทดสอบทำนายข้อมูลตัวอย่าง 15 แถวแรก ---
 Sample_No  Excel_No       Location Actual_Crop Predicted_Crop  Confidence (%)  Match
         1      1611   SAKON NAKHON      orange         orange       99.459999   True
         2      1074   MAE HONG SON      banana         banana       99.599998   True
         3      1914     NARATHIWAT      cotton         cotton       99.300003   True
         4       102      UTTARADIT       maize          maize       99.809998   True
         5      1647      UTTARADIT      orange         orange       99.669998   True
         6       223   PRACHIN BURI    chickpea       chickpea       99.580002   True
         7        30         ROI ET        rice           rice       99.589996   True
         8       703        LAMPHUN   blackgram      blackgram       99.669998   True
         9      1099        KALASIN      banana         banana       99.349998   True
        10      1640 KAMPHAENG PHET      orange         orange       99.599998   True
     